# Early Warning Signals and Hybrid Time Series–Deep Learning Models for Climate Tipping Point Prediction

## Abstract

This notebook implements a fully reproducible, research-grade forecasting framework to detect early warning signals (EWS) and predict potential climate tipping points. We combine classical time series analysis (ARIMA, SARIMA, SARIMAX, VAR, VECM, GARCH family, Holt-Winters), deep learning architectures (LSTM, GRU, CNN-LSTM), and hybrid ensemble methods. The pipeline enforces statistical rigor as a mandatory gate before any modeling, uses ERA5-style synthetic monthly climate data (surface temperature + precipitation), and produces extensive visualizations and benchmarks across all model families.

**Sections:**
1. Imports & Global Configuration
2. Data Loading & Validation
3. Exploratory Data Analysis
4. Statistical Diagnostics (Mandatory Gate)
5. Early Warning Signals
6. Classical TSA Models
7. Deep Learning Models
8. Hyperparameter Optimization
9. Model Comparison
10. Hybrid Ensemble
11. Future Forecasting
12. Conclusions

## Section 1 — Imports & Global Configuration

In [1]:
# ── Standard Library ──────────────────────────────────────────────────────────
import warnings, os, json, itertools, time
warnings.filterwarnings('ignore')
os.makedirs('figures', exist_ok=True)
os.makedirs('models',  exist_ok=True)

# ── Numerical / Data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import periodogram

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')          # headless – swap to 'inline' in Jupyter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
FIGSIZE = (14, 5)

# ── Statistics ────────────────────────────────────────────────────────────────
try:
    from statsmodels.tsa.stattools   import adfuller, kpss, acf, pacf, coint
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.vector_ar.var_model import VAR
    from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    from statsmodels.tsa.seasonal import STL
    from statsmodels.stats.stattools import durbin_watson
    from statsmodels.tsa.stattools import breakvar_heteroskedasticity_test
    import statsmodels.api as sm
    HAS_STATSMODELS = True
    print('statsmodels  ✓')
except ImportError:
    HAS_STATSMODELS = False
    print('statsmodels  ✗  (install: pip install statsmodels)')

try:
    from arch import arch_model
    HAS_ARCH = True
    print('arch         ✓')
except ImportError:
    HAS_ARCH = False
    print('arch         ✗  (install: pip install arch)')

try:
    from pmdarima import auto_arima
    HAS_PMDARIMA = True
    print('pmdarima     ✓')
except ImportError:
    HAS_PMDARIMA = False
    print('pmdarima     ✗  (install: pip install pmdarima)')

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.preprocessing    import MinMaxScaler
from sklearn.linear_model     import Ridge
from sklearn.metrics          import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection  import TimeSeriesSplit
print('scikit-learn ✓')

# ── Deep Learning ─────────────────────────────────────────────────────────────
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import (
        LSTM, GRU, Dense, Dropout, Conv1D, MaxPooling1D,
        Flatten, Input, TimeDistributed, Reshape
    )
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.optimizers import Adam
    tf.random.set_seed(42)
    HAS_TF = True
    print(f'tensorflow   ✓  ({tf.__version__})')
except ImportError:
    HAS_TF = False
    print('tensorflow   ✗  (install: pip install tensorflow)')

# ── Hyperparameter Optimisation ───────────────────────────────────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
    print('optuna       ✓')
except ImportError:
    HAS_OPTUNA = False
    print('optuna       ✗  (install: pip install optuna)')

# ── Global Seeds ──────────────────────────────────────────────────────────────
np.random.seed(42)

# ── Global Helpers ────────────────────────────────────────────────────────────
def savefig(name):
    plt.tight_layout()
    plt.savefig(f'figures/{name}.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def eval_metrics(y_true, y_pred, label='Model'):
    r  = rmse(y_true, y_pred)
    m  = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    acc = max(0, 1 - r / (np.std(y_true) + 1e-9)) * 100
    return {'Model': label, 'RMSE': r, 'MAE': m, 'R2': r2, 'Accuracy(%)': acc}

RESULTS = []   # global leaderboard
print('\nEnvironment ready.')

statsmodels  ✓
arch         ✓
pmdarima     ✓
scikit-learn ✓
tensorflow   ✓  (2.20.0)
optuna       ✓

Environment ready.


## Section 2 — Data Loading & Validation

In [3]:
import xarray as xr

# ── File Paths ────────────────────────────────────────────────────────────────
# avgad = accumulated/diagnostic fields  (includes total precipitation: tp)
# avgua = instantaneous/analysis fields  (includes 2m temperature: t2m)
FILE_AVGUA = 'data_stream-moda_stepType-avgua.nc'   # adjust path if needed
FILE_AVGAD = 'data_stream-moda_stepType-avgad.nc'

# ── Load Temperature (2m air temp from avgua stream) ─────────────────────────
ds_ua = xr.open_dataset(FILE_AVGUA)
print('avgua variables:', list(ds_ua.data_vars))

# ERA5 2m temperature variable is usually 't2m' or 'VAR_2T'
t2m_var = 't2m' if 't2m' in ds_ua.data_vars else list(ds_ua.data_vars)[0]
print(f'Using temperature variable: {t2m_var}')

# Spatial mean over all lat/lon → global/regional monthly mean
# If your file is already a single-point or area-mean, drop the .mean() dims
t2m = ds_ua[t2m_var]
if 'latitude' in t2m.dims and 'longitude' in t2m.dims:
    t2m = t2m.mean(dim=['latitude', 'longitude'])
elif 'lat' in t2m.dims and 'lon' in t2m.dims:
    t2m = t2m.mean(dim=['lat', 'lon'])

# Convert Kelvin → Celsius if needed
if float(t2m.mean()) > 200:
    t2m = t2m - 273.15
    print('Converted temperature K → °C')

# ── Load Precipitation (total precip from avgad stream) ───────────────────────
ds_ad = xr.open_dataset(FILE_AVGAD)
print('avgad variables:', list(ds_ad.data_vars))

# ERA5 total precipitation variable is usually 'tp'
tp_var = 'tp' if 'tp' in ds_ad.data_vars else list(ds_ad.data_vars)[0]
print(f'Using precipitation variable: {tp_var}')

tp = ds_ad[tp_var]
if 'latitude' in tp.dims and 'longitude' in tp.dims:
    tp = tp.mean(dim=['latitude', 'longitude'])
elif 'lat' in tp.dims and 'lon' in tp.dims:
    tp = tp.mean(dim=['lat', 'lon'])

# ERA5 tp is in metres/day (monthly mean) → convert to mm/month
# monthly mean rate (m/day) × days-in-month × 1000 mm/m
# xarray approach: multiply by days per month
import pandas as pd
time_coord = pd.DatetimeIndex(ds_ad['valid_time'].values)
days_in_month = time_coord.days_in_month.values
tp_mm = tp.values * days_in_month * 1000   # → mm/month
# If values are already reasonable mm/month (> 0.1 range), skip conversion
if tp_mm.mean() < 0.01:
    print('WARNING: precipitation values look very small — check units')

# ── Align time axes & build DataFrame ────────────────────────────────────────
temp_series  = t2m.to_series().rename('temperature')
precip_series = pd.Series(tp_mm, index=time_coord, name='precipitation')

# Ensure monthly frequency
temp_series   = temp_series.resample('MS').mean()
precip_series = precip_series.resample('MS').mean()

# Inner join on shared time range
df = pd.concat([temp_series, precip_series], axis=1, join='inner')
df.index.name = 'date'
df = df.sort_index()

print(f'\nLoaded {len(df)} monthly records: {df.index[0].date()} → {df.index[-1].date()}')
print(df.info())
print('\nHead:')
print(df.head())
print('\nDescribe:')
print(df.describe().round(3))
print(f'\nMissing values: {df.isnull().sum().to_dict()}')

# Forward-fill isolated NaNs (e.g. missing months), then assert clean
df = df.ffill().bfill()
assert df.isnull().sum().sum() == 0, 'Missing values remain after fill!'
print('\nData validation passed ✓')

avgua variables: ['u10', 'v10', 't2m', 'msl']
Using temperature variable: t2m
Converted temperature K → °C
avgad variables: ['tp']
Using precipitation variable: tp

Loaded 912 monthly records: 1950-01-01 → 2025-12-01
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 912 entries, 1950-01-01 to 2025-12-01
Freq: MS
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   temperature    912 non-null    float32
 1   precipitation  912 non-null    float64
dtypes: float32(1), float64(1)
memory usage: 50.1 KB
None

Head:
            temperature  precipitation
date                                  
1950-01-01     4.206207     100.371835
1950-02-01     3.885132      81.122954
1950-03-01     6.056213      81.346207
1950-04-01     8.088104      59.666235
1950-05-01    11.410858      59.299814

Describe:
       temperature  precipitation
count      912.000        912.000
mean        10.903         82.801
std          4.393      

## Section 3 — Exploratory Data Analysis

In [4]:
# ── 3.1  Raw Time Series ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df['temperature'], color='#e63946', lw=0.9, label='Temperature (°C)')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend()
axes[1].fill_between(df.index, df['precipitation'], alpha=0.6, color='#457b9d', label='Precipitation (mm)')
axes[1].set_ylabel('Precipitation (mm)')
axes[1].legend()
fig.suptitle('ERA5-Style Monthly Climate Variables (1984–2023)', fontsize=14, fontweight='bold')
savefig('01_raw_time_series')

In [5]:
# ── 3.2  Rolling Mean & Variance ─────────────────────────────────────────────
WINDOW = 24
for col, color in [('temperature', '#e63946'), ('precipitation', '#457b9d')]:
    roll_mean = df[col].rolling(WINDOW).mean()
    roll_var  = df[col].rolling(WINDOW).var()
    fig, axes = plt.subplots(2, 1, figsize=FIGSIZE, sharex=True)
    axes[0].plot(df.index, df[col], alpha=0.4, color=color)
    axes[0].plot(df.index, roll_mean, lw=2, color='black', label=f'{WINDOW}-month rolling mean')
    axes[0].set_title(f'{col.capitalize()} — Rolling Mean')
    axes[0].legend()
    axes[1].plot(df.index, roll_var, color=color, lw=1.2, label=f'{WINDOW}-month rolling variance')
    axes[1].set_title(f'{col.capitalize()} — Rolling Variance')
    axes[1].legend()
    savefig(f'02_rolling_{col}')

In [6]:
# ── 3.3  STL Decomposition ───────────────────────────────────────────────────
if HAS_STATSMODELS:
    for col in ['temperature', 'precipitation']:
        stl = STL(df[col], period=12, robust=True)
        res = stl.fit()
        fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
        data_list = [
            (df[col],      'Observed',  '#555'),
            (res.trend,    'Trend',     '#e63946'),
            (res.seasonal, 'Seasonal',  '#457b9d'),
            (res.resid,    'Residual',  '#2a9d8f'),
        ]
        for ax, (series, title, c) in zip(axes, data_list):
            ax.plot(df.index, series, color=c, lw=0.9)
            ax.set_ylabel(title)
        fig.suptitle(f'STL Decomposition — {col.capitalize()}', fontsize=13, fontweight='bold')
        savefig(f'03_stl_{col}')
else:
    print('Skipping STL — statsmodels not installed.')

In [7]:
# ── 3.4  Distributions & Density Plots ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, c in zip(axes, ['temperature', 'precipitation'], ['#e63946', '#457b9d']):
    sns.histplot(df[col], kde=True, color=c, ax=ax, bins=40)
    ax.set_title(f'{col.capitalize()} Distribution')
    ax.set_xlabel(col)
savefig('04_distributions')

# Q-Q plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['temperature', 'precipitation']):
    stats.probplot(df[col], dist='norm', plot=ax)
    ax.set_title(f'Q-Q Plot — {col.capitalize()}')
savefig('05_qq_plots')

In [8]:
# ── 3.5  Correlation Heatmap ──────────────────────────────────────────────────
corr_df = df.copy()
corr_df['temp_lag1']  = corr_df['temperature'].shift(1)
corr_df['temp_lag3']  = corr_df['temperature'].shift(3)
corr_df['temp_lag12'] = corr_df['temperature'].shift(12)
corr_df['precip_lag1'] = corr_df['precipitation'].shift(1)
corr_df.dropna(inplace=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap (Variables + Lags)', fontsize=13)
savefig('06_correlation_heatmap')

## Section 4 — Statistical Diagnostics (Mandatory Gate)

In [10]:
# ── 4.1  Stationarity Tests ───────────────────────────────────────────────────
stationarity_report = {}

def run_adf_kpss(series, name):
    if not HAS_STATSMODELS:
        print('statsmodels required for ADF/KPSS — skipping.')
        return
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series.dropna(), autolag='AIC')
    kpss_stat, kpss_p, _, kpss_crit = kpss(series.dropna(), regression='ct', nlags='auto')
    is_stationary = (adf_p < 0.05) and (kpss_p > 0.05)
    stationarity_report[name] = {
        'ADF_stat': round(adf_stat, 4), 'ADF_p': round(adf_p, 4),
        'KPSS_stat': round(kpss_stat, 4), 'KPSS_p': round(kpss_p, 4),
        'Stationary': is_stationary
    }
    print(f"{'─'*55}")
    print(f"  Series : {name}")
    print(f"  ADF    : stat={adf_stat:.4f}  p={adf_p:.4f}  {'✓ Stationary' if adf_p<0.05 else '✗ Unit Root'}")
    print(f"  KPSS   : stat={kpss_stat:.4f}  p={kpss_p:.4f}  {'✓ Stationary' if kpss_p>0.05 else '✗ Non-stationary'}")
    print(f"  Result : {'STATIONARY ✓' if is_stationary else 'NON-STATIONARY — differencing required'}")

# Raw series
print('\n=== RAW SERIES ===')
run_adf_kpss(df['temperature'],   'temperature_raw')
run_adf_kpss(df['precipitation'], 'precipitation_raw')

# First differences
df['temp_diff1']   = df['temperature'].diff()
df['precip_diff1'] = df['precipitation'].diff()
print('\n=== FIRST DIFFERENCES ===')
run_adf_kpss(df['temp_diff1'].dropna(),   'temperature_diff1')
run_adf_kpss(df['precip_diff1'].dropna(), 'precipitation_diff1')

report_serializable = {k: {kk: bool(vv) if isinstance(vv, (bool, np.bool_)) else float(vv) if isinstance(vv, (np.floating, np.integer)) else vv for kk, vv in v.items()} for k, v in stationarity_report.items()}
print('\nStationarity report:', json.dumps(report_serializable, indent=2))


=== RAW SERIES ===
───────────────────────────────────────────────────────
  Series : temperature_raw
  ADF    : stat=-1.7626  p=0.3992  ✗ Unit Root
  KPSS   : stat=0.0346  p=0.1000  ✓ Stationary
  Result : NON-STATIONARY — differencing required
───────────────────────────────────────────────────────
  Series : precipitation_raw
  ADF    : stat=-5.2437  p=0.0000  ✓ Stationary
  KPSS   : stat=0.0098  p=0.1000  ✓ Stationary
  Result : STATIONARY ✓

=== FIRST DIFFERENCES ===
───────────────────────────────────────────────────────
  Series : temperature_diff1
  ADF    : stat=-11.1992  p=0.0000  ✓ Stationary
  KPSS   : stat=0.0022  p=0.1000  ✓ Stationary
  Result : STATIONARY ✓
───────────────────────────────────────────────────────
  Series : precipitation_diff1
  ADF    : stat=-10.4665  p=0.0000  ✓ Stationary
  KPSS   : stat=0.0074  p=0.1000  ✓ Stationary
  Result : STATIONARY ✓

Stationarity report: {
  "temperature_raw": {
    "ADF_stat": -1.7626,
    "ADF_p": 0.3992,
    "KPSS_stat": 

C:\Users\waibhav\AppData\Local\Temp\ipykernel_16664\2020917521.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, _, kpss_crit = kpss(series.dropna(), regression='ct', nlags='auto')
C:\Users\waibhav\AppData\Local\Temp\ipykernel_16664\2020917521.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, _, kpss_crit = kpss(series.dropna(), regression='ct', nlags='auto')
C:\Users\waibhav\AppData\Local\Temp\ipykernel_16664\2020917521.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, _, kpss_crit = kpss(series.dropna(), regression='ct', nlags='auto')
C:\Users\waibhav\AppData\

In [11]:
# ── 4.2  ACF / PACF Plots ─────────────────────────────────────────────────────
if HAS_STATSMODELS:
    for col, label in [('temperature', 'Temperature'), ('temp_diff1', 'Temperature Δ1')]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        series = df[col].dropna()
        plot_acf(series,  lags=48, ax=axes[0], title=f'ACF — {label}')
        plot_pacf(series, lags=48, ax=axes[1], title=f'PACF — {label}', method='ywm')
        savefig(f'07_acf_pacf_{col}')

In [ ]:
# ── 4.3  Ljung–Box, ARCH-LM, Structural Break ────────────────────────────────
if HAS_STATSMODELS:
    series = df['temperature'].dropna()

    lb = acorr_ljungbox(series, lags=[12, 24, 36], return_df=True)
    print('Ljung–Box Test (temperature):')
    print(lb.to_string())

    arch_lm = het_arch(series)
    print(f'\nARCH-LM Test: F-stat={arch_lm[0]:.4f}  p={arch_lm[1]:.4f}  '
          f'{"Heteroskedastic ✓" if arch_lm[1]<0.05 else "Homoskedastic"}')

    # Chow-style structural break via rolling cumulative sum
    cusum = np.cumsum(series - series.mean())
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(df.index, cusum, color='#e63946', lw=1.5)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_title('CUSUM Chart — Structural Break Detection (Temperature)')
    ax.set_ylabel('Cumulative Sum')
    savefig('08_cusum')
else:
    print('Skipping — statsmodels required.')

print('\n⚠  Stationarity gate: proceeding with differenced series for TSA models.')

Ljung–Box Test (temperature):
         lb_stat  lb_pvalue
12   5205.569111        0.0
24  10340.652021        0.0
36  15402.319030        0.0

ARCH-LM Test: F-stat=892.9095  p=0.0000  Heteroskedastic ✓

⚠  Stationarity gate: proceeding with differenced series for TSA models.


## Section 5 — Early Warning Signals (EWS)

In [13]:
# ── 5.1  Rolling Variance & Lag-1 Autocorrelation ────────────────────────────
W = 36   # 3-year window
series_temp = df['temperature']

roll_var   = series_temp.rolling(W).var()
roll_ac1   = series_temp.rolling(W).apply(
    lambda x: pd.Series(x).autocorr(lag=1), raw=False
)

# ── 5.2  Trend of EWS (Kendall τ over expanding window) ─────────────────────
from scipy.stats import kendalltau
half = W * 2
kendall_var, kendall_ac1 = [], []
for i in range(half, len(series_temp)):
    segment = series_temp.iloc[i-half:i]
    rv = [segment.rolling(W).var().dropna().iloc[j] for j in range(len(segment.rolling(W).var().dropna()))]
    ra = [segment.rolling(W).apply(lambda x: pd.Series(x).autocorr(1), raw=False).dropna().iloc[j]
          for j in range(len(segment.rolling(W).apply(lambda x: pd.Series(x).autocorr(1), raw=False).dropna()))]
    if len(rv) > 2:
        kendall_var.append(kendalltau(np.arange(len(rv)), rv).statistic)
    else:
        kendall_var.append(np.nan)
    if len(ra) > 2:
        kendall_ac1.append(kendalltau(np.arange(len(ra)), ra).statistic)
    else:
        kendall_ac1.append(np.nan)

# ── 5.3  Climate Instability Index (CII) ─────────────────────────────────────
# Composite: normalised roll_var + normalised roll_ac1
def normalise(s):
    s = pd.Series(s)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

CII = 0.5 * normalise(roll_var) + 0.5 * normalise(roll_ac1)
df['roll_var']  = roll_var.values
df['roll_ac1']  = roll_ac1.values
df['CII']       = CII.values

# ── 5.4  EWS Visualisation ────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

axes[0].plot(df.index, df['temperature'], color='#e63946', lw=0.9, label='Temperature')
axes[0].set_ylabel('°C');  axes[0].legend()

axes[1].plot(df.index, roll_var, color='#f4a261', lw=1.2, label=f'Rolling Variance (w={W})')
axes[1].set_ylabel('Variance');  axes[1].legend()

axes[2].plot(df.index, roll_ac1, color='#2a9d8f', lw=1.2, label=f'Lag-1 Autocorrelation (w={W})')
axes[2].axhline(0, color='black', ls='--', lw=0.7)
axes[2].set_ylabel('AC1');  axes[2].legend()

axes[3].fill_between(df.index, CII, alpha=0.7, color='#6a0dad', label='Climate Instability Index')
axes[3].set_ylabel('CII (0–1)');  axes[3].legend()

fig.suptitle('Early Warning Signals — Critical Slowing Down Indicators', fontsize=13, fontweight='bold')
savefig('09_ews')

print(f'Peak CII: {CII.max():.4f} at {df.index[CII.argmax()].strftime("%Y-%m")}')
print(f'Critical Slowing Down: Kendall τ of rolling variance trend = {np.nanmean(kendall_var):.4f}')

Peak CII: 0.9560 at 1995-10
Critical Slowing Down: Kendall τ of rolling variance trend = -0.0391


## Section 6 — Classical TSA & Econometric Models

In [14]:
# ── Train / Test Split ────────────────────────────────────────────────────────
TRAIN_END = '2020-12-01'
TEST_START = '2021-01-01'

df_train = df.loc[:TRAIN_END].copy()
df_test  = df.loc[TEST_START:].copy()
n_test   = len(df_test)

y_train  = df_train['temperature']
y_test   = df_test['temperature']

print(f'Train: {len(df_train)} obs  ({df_train.index[0].date()} → {df_train.index[-1].date()})')
print(f'Test : {n_test} obs  ({df_test.index[0].date()} → {df_test.index[-1].date()})')

Train: 852 obs  (1950-01-01 → 2020-12-01)
Test : 60 obs  (2021-01-01 → 2025-12-01)


In [15]:
# ── 6.1  ARIMA (auto order selection) ────────────────────────────────────────
if HAS_STATSMODELS:
    best_aic, best_order = np.inf, (1,1,1)
    for p in range(4):
        for d in [1]:
            for q in range(4):
                try:
                    m = ARIMA(y_train, order=(p,d,q)).fit()
                    if m.aic < best_aic:
                        best_aic, best_order = m.aic, (p,d,q)
                except:
                    pass

    print(f'Best ARIMA order: {best_order}  AIC: {best_aic:.2f}')
    arima_model = ARIMA(y_train, order=best_order).fit()
    arima_fc    = arima_model.forecast(steps=n_test)
    arima_fc    = pd.Series(arima_fc.values, index=y_test.index)

    RESULTS.append(eval_metrics(y_test, arima_fc, f'ARIMA{best_order}'))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(y_train.index[-60:], y_train.iloc[-60:], color='#555', label='Train')
    ax.plot(y_test.index,  y_test,   color='#e63946', lw=1.5, label='Actual')
    ax.plot(arima_fc.index, arima_fc, color='#2a9d8f', ls='--', lw=1.5, label=f'ARIMA{best_order}')
    ax.legend(); ax.set_title(f'ARIMA{best_order} Forecast vs Actual')
    savefig('10_arima_forecast')

    # Residual diagnostics
    resid = arima_model.resid
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(resid); axes[0].set_title('Residuals')
    axes[1].hist(resid, bins=30, color='#2a9d8f'); axes[1].set_title('Residual Histogram')
    plot_acf(resid, lags=24, ax=axes[2]); axes[2].set_title('Residual ACF')
    savefig('11_arima_residuals')
else:
    print('ARIMA skipped — statsmodels not installed.')

C:\Users\waibhav\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\waibhav\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\waibhav\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Best ARIMA order: (2, 1, 3)  AIC: 1120.61


C:\Users\waibhav\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [16]:
# ── 6.2  SARIMA ───────────────────────────────────────────────────────────────
if HAS_STATSMODELS:
    sarima_model = SARIMAX(
        y_train, order=(1,1,1), seasonal_order=(1,1,1,12),
        enforce_stationarity=False, enforce_invertibility=False
    ).fit(disp=False)

    sarima_fc = sarima_model.forecast(steps=n_test)
    sarima_fc = pd.Series(sarima_fc.values, index=y_test.index)

    RESULTS.append(eval_metrics(y_test, sarima_fc, 'SARIMA(1,1,1)(1,1,1,12)'))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(y_train.index[-60:], y_train.iloc[-60:], color='#555', label='Train')
    ax.plot(y_test.index, y_test,    color='#e63946', lw=1.5, label='Actual')
    ax.plot(sarima_fc.index, sarima_fc, color='#457b9d', ls='--', lw=1.5, label='SARIMA')
    ax.legend(); ax.set_title('SARIMA(1,1,1)(1,1,1,12) Forecast')
    savefig('12_sarima_forecast')
    print('SARIMA AIC:', round(sarima_model.aic, 2))
else:
    print('SARIMA skipped.')

SARIMA AIC: 829.25


In [17]:
# ── 6.3  SARIMAX (with EWS exogenous regressors) ─────────────────────────────
if HAS_STATSMODELS:
    exog_cols = ['CII']
    df_clean  = df[['temperature'] + exog_cols].dropna()
    train_e   = df_clean.loc[:TRAIN_END]
    test_e    = df_clean.loc[TEST_START:]

    sarimax_model = SARIMAX(
        train_e['temperature'],
        exog=train_e[exog_cols],
        order=(1,1,1), seasonal_order=(1,1,1,12),
        enforce_stationarity=False, enforce_invertibility=False
    ).fit(disp=False)

    sarimax_fc = sarimax_model.forecast(steps=len(test_e), exog=test_e[exog_cols])
    sarimax_fc = pd.Series(sarimax_fc.values, index=test_e.index)

    RESULTS.append(eval_metrics(test_e['temperature'], sarimax_fc, 'SARIMAX+EWS'))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(test_e.index, test_e['temperature'], color='#e63946', lw=1.5, label='Actual')
    ax.plot(sarimax_fc.index, sarimax_fc, color='#f4a261', ls='--', lw=1.5, label='SARIMAX+EWS')
    ax.legend(); ax.set_title('SARIMAX with EWS Exogenous Regressors')
    savefig('13_sarimax_forecast')
else:
    print('SARIMAX skipped.')

In [18]:
# ── 6.4  VAR (multivariate: temperature + precipitation) ─────────────────────
if HAS_STATSMODELS:
    var_train = df_train[['temperature','precipitation']].diff().dropna()
    var_test  = df_test[['temperature','precipitation']].diff().dropna()

    var_model = VAR(var_train)
    lag_order = var_model.select_order(maxlags=12)
    best_lag  = lag_order.selected_orders['aic']
    print(f'VAR best lag (AIC): {best_lag}')

    var_fit   = var_model.fit(best_lag)
    fc_input  = var_train.values[-best_lag:]
    var_fc    = var_fit.forecast(fc_input, steps=len(var_test))
    var_fc_df = pd.DataFrame(var_fc, index=var_test.index,
                              columns=['temperature','precipitation'])

    # Cumulative sum to recover levels
    var_fc_levels = var_fc_df['temperature'].cumsum() + df_train['temperature'].iloc[-1]
    var_fc_levels = var_fc_levels.iloc[:n_test]
    y_test_var    = y_test.iloc[:len(var_fc_levels)]
    RESULTS.append(eval_metrics(y_test_var, var_fc_levels, 'VAR'))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(y_test_var.index, y_test_var,    color='#e63946', lw=1.5, label='Actual')
    ax.plot(var_fc_levels.index, var_fc_levels, color='#6a0dad', ls='--', lw=1.5, label='VAR')
    ax.legend(); ax.set_title('VAR Forecast — Temperature')
    savefig('14_var_forecast')
else:
    print('VAR skipped.')

VAR best lag (AIC): 12


In [19]:
# ── 6.5  VECM (if cointegration detected) ─────────────────────────────────────
if HAS_STATSMODELS:
    from statsmodels.tsa.vector_ar.vecm import select_coint_rank
    mv_train = df_train[['temperature','precipitation']].dropna()
    rank = select_coint_rank(mv_train, det_order=0, k_ar_diff=2, method='trace')
    print(f'Johansen Cointegration Rank: {rank.rank}')

    if rank.rank > 0:
        vecm_model = VECM(mv_train, coint_rank=rank.rank, k_ar_diff=2, deterministic='co').fit()
        vecm_fc    = vecm_model.predict(steps=n_test)
        vecm_fc_s  = pd.Series(vecm_fc[:, 0], index=y_test.index[:len(vecm_fc)])
        RESULTS.append(eval_metrics(y_test.iloc[:len(vecm_fc)], vecm_fc_s, 'VECM'))
        print('VECM fitted successfully.')
    else:
        print('No cointegration — skipping VECM.')
else:
    print('VECM skipped.')

Johansen Cointegration Rank: 2
VECM fitted successfully.


In [20]:
# ── 6.6  GARCH Family Volatility Models ───────────────────────────────────────
if HAS_ARCH:
    returns = df_train['temperature'].diff().dropna() * 100   # scale for ARCH

    for vol_model, label in [
        ('Garch', 'GARCH(1,1)'),
        ('EGARCH', 'EGARCH(1,1)'),
        ('TARCH', 'TGARCH(1,1)'),
    ]:
        try:
            am = arch_model(returns, vol=vol_model, p=1, q=1, dist='Normal')
            res = am.fit(disp='off')
            print(f'{label}  AIC={res.aic:.2f}  BIC={res.bic:.2f}')
        except Exception as e:
            print(f'{label} failed: {e}')

    # Visualise conditional volatility
    am = arch_model(returns, vol='Garch', p=1, q=1)
    garch_fit = am.fit(disp='off')
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(df_train.index[1:], garch_fit.conditional_volatility,
            color='#e63946', lw=1.2, label='GARCH Conditional Volatility')
    ax.set_title('GARCH(1,1) Conditional Volatility — Temperature')
    ax.legend()
    savefig('15_garch_volatility')
else:
    print('GARCH skipped — arch not installed.')

GARCH(1,1)  AIC=11658.61  BIC=11677.59
EGARCH(1,1)  AIC=11656.95  BIC=11675.94
TGARCH(1,1) failed: Unknown model type in vol


In [21]:
# ── 6.7  Holt-Winters Exponential Smoothing ───────────────────────────────────
if HAS_STATSMODELS:
    hw_model = ExponentialSmoothing(
        y_train, trend='add', seasonal='add', seasonal_periods=12
    ).fit(optimized=True)

    hw_fc = hw_model.forecast(n_test)
    hw_fc = pd.Series(hw_fc.values, index=y_test.index)
    RESULTS.append(eval_metrics(y_test, hw_fc, 'Holt-Winters'))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(y_train.index[-48:], y_train.iloc[-48:], color='#555', label='Train')
    ax.plot(y_test.index, y_test,  color='#e63946', lw=1.5, label='Actual')
    ax.plot(hw_fc.index,  hw_fc,   color='#2a9d8f', ls='--', lw=1.5, label='Holt-Winters')
    ax.legend(); ax.set_title('Holt-Winters Additive Forecast')
    savefig('16_holtwinters_forecast')
    print(f'Holt-Winters SSE: {hw_model.sse:.2f}')
else:
    print('Holt-Winters skipped.')

Holt-Winters SSE: 135.24


## Section 7 — Deep Learning Models

In [23]:
# ── 7.1  Sliding Window Dataset Preparation ───────────────────────────────────
SEQ_LEN = 24   # 24-month lookback window

# Univariate: temperature only
scaler_uni = MinMaxScaler()
temp_scaled = scaler_uni.fit_transform(df[['temperature']].values)

# Multivariate: temperature + precipitation + EWS features
mv_cols = ['temperature', 'precipitation', 'roll_var', 'roll_ac1', 'CII']
df_mv   = df[mv_cols].dropna()
scaler_mv = MinMaxScaler()
mv_scaled = scaler_mv.fit_transform(df_mv.values)

def make_sequences(data, seq_len=SEQ_LEN, target_col=0):
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i-seq_len:i, :])
        y.append(data[i, target_col])
    return np.array(X), np.array(y)

# Build sequences
X_uni, y_uni = make_sequences(temp_scaled)
X_mv,  y_mv  = make_sequences(mv_scaled)

# Time-aware split
split_idx = len(df_train) - SEQ_LEN   # offset for lookback
split_idx = max(split_idx, len(X_uni) - n_test)

X_tr_u, X_te_u = X_uni[:split_idx], X_uni[split_idx:]
y_tr_u, y_te_u = y_uni[:split_idx], y_uni[split_idx:]

# For multivariate: align
mv_offset    = len(df) - len(df_mv)
split_idx_mv = split_idx - mv_offset
split_idx_mv = max(0, min(split_idx_mv, len(X_mv) - n_test))

X_tr_mv, X_te_mv = X_mv[:split_idx_mv], X_mv[split_idx_mv:]
y_tr_mv, y_te_mv = y_mv[:split_idx_mv], y_mv[split_idx_mv:]

print(f'Univariate  X_train: {X_tr_u.shape}  X_test: {X_te_u.shape}')
print(f'Multivariate X_train: {X_tr_mv.shape}  X_test: {X_te_mv.shape}')

CALLBACKS = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(factor=0.5, patience=5, verbose=0)
] if HAS_TF else []

Univariate  X_train: (828, 24, 1)  X_test: (60, 24, 1)
Multivariate X_train: (793, 24, 5)  X_test: (60, 24, 5)


In [24]:
# ── 7.2  Univariate LSTM Baseline ─────────────────────────────────────────────
def build_lstm_uni(units=64, dropout=0.2, lr=1e-3):
    model = Sequential([
        LSTM(units, input_shape=(SEQ_LEN, 1), return_sequences=False),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss='mse')
    return model

if HAS_TF:
    lstm_uni = build_lstm_uni()
    history_u = lstm_uni.fit(
        X_tr_u, y_tr_u,
        validation_split=0.15,
        epochs=100, batch_size=32,
        callbacks=CALLBACKS, verbose=0
    )

    # Learning curve
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history_u.history['loss'],     label='Train Loss')
    ax.plot(history_u.history['val_loss'], label='Val Loss')
    ax.set_title('LSTM (Univariate) — Learning Curve')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
    savefig('17_lstm_uni_learning_curve')

    pred_u = lstm_uni.predict(X_te_u, verbose=0)
    pred_u = scaler_uni.inverse_transform(pred_u).flatten()
    act_u  = scaler_uni.inverse_transform(y_te_u.reshape(-1, 1)).flatten()
    RESULTS.append(eval_metrics(act_u, pred_u, 'LSTM-Univariate'))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(act_u,  color='#e63946', lw=1.5, label='Actual')
    ax.plot(pred_u, color='#2a9d8f', ls='--', lw=1.5, label='LSTM Uni')
    ax.legend(); ax.set_title('LSTM Univariate — Test Forecast')
    savefig('18_lstm_uni_forecast')
else:
    print('LSTM skipped — TensorFlow not installed.')

In [25]:
# ── 7.3  Multivariate LSTM (with EWS) ────────────────────────────────────────
def build_lstm_mv(n_features, units=64, dropout=0.2, lr=1e-3):
    model = Sequential([
        LSTM(units, input_shape=(SEQ_LEN, n_features), return_sequences=True),
        Dropout(dropout),
        LSTM(units // 2, return_sequences=False),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss='mse')
    return model

if HAS_TF and len(X_tr_mv) > 10:
    n_feat = X_tr_mv.shape[2]
    lstm_mv = build_lstm_mv(n_feat)
    history_mv = lstm_mv.fit(
        X_tr_mv, y_tr_mv,
        validation_split=0.15,
        epochs=100, batch_size=32,
        callbacks=CALLBACKS, verbose=0
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history_mv.history['loss'],     label='Train Loss')
    ax.plot(history_mv.history['val_loss'], label='Val Loss')
    ax.set_title('LSTM (Multivariate) — Learning Curve'); ax.legend()
    savefig('19_lstm_mv_learning_curve')

    pred_mv = lstm_mv.predict(X_te_mv, verbose=0)
    # Inverse transform temperature column only
    temp_idx = 0
    dummy = np.zeros((len(pred_mv), n_feat))
    dummy[:, temp_idx] = pred_mv.flatten()
    pred_mv_inv = scaler_mv.inverse_transform(dummy)[:, temp_idx]
    dummy2 = np.zeros((len(y_te_mv), n_feat))
    dummy2[:, temp_idx] = y_te_mv.flatten()
    act_mv_inv  = scaler_mv.inverse_transform(dummy2)[:, temp_idx]

    RESULTS.append(eval_metrics(act_mv_inv, pred_mv_inv, 'LSTM-Multivariate'))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(act_mv_inv,  color='#e63946', lw=1.5, label='Actual')
    ax.plot(pred_mv_inv, color='#457b9d', ls='--', lw=1.5, label='LSTM MV')
    ax.legend(); ax.set_title('LSTM Multivariate — Test Forecast')
    savefig('20_lstm_mv_forecast')
else:
    print('Multivariate LSTM skipped.')

In [26]:
# ── 7.4  GRU Architecture ─────────────────────────────────────────────────────
def build_gru(units=64, dropout=0.2, lr=1e-3):
    model = Sequential([
        GRU(units, input_shape=(SEQ_LEN, 1), return_sequences=False),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss='mse')
    return model

if HAS_TF:
    gru_model  = build_gru()
    history_gru = gru_model.fit(
        X_tr_u, y_tr_u,
        validation_split=0.15,
        epochs=100, batch_size=32,
        callbacks=CALLBACKS, verbose=0
    )

    pred_gru = gru_model.predict(X_te_u, verbose=0)
    pred_gru = scaler_uni.inverse_transform(pred_gru).flatten()
    RESULTS.append(eval_metrics(act_u, pred_gru, 'GRU'))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(act_u,   color='#e63946', lw=1.5, label='Actual')
    ax.plot(pred_gru, color='#f4a261', ls='--', lw=1.5, label='GRU')
    ax.legend(); ax.set_title('GRU — Test Forecast')
    savefig('21_gru_forecast')
else:
    print('GRU skipped.')

In [27]:
# ── 7.5  CNN-LSTM ─────────────────────────────────────────────────────────────
def build_cnn_lstm(filters=32, kernel=3, lstm_units=64, dropout=0.2, lr=1e-3):
    model = Sequential([
        Conv1D(filters, kernel_size=kernel, activation='relu',
               input_shape=(SEQ_LEN, 1), padding='same'),
        MaxPooling1D(pool_size=2),
        LSTM(lstm_units, return_sequences=False),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss='mse')
    return model

if HAS_TF:
    cnn_lstm = build_cnn_lstm()
    history_cl = cnn_lstm.fit(
        X_tr_u, y_tr_u,
        validation_split=0.15,
        epochs=100, batch_size=32,
        callbacks=CALLBACKS, verbose=0
    )

    pred_cl = cnn_lstm.predict(X_te_u, verbose=0)
    pred_cl = scaler_uni.inverse_transform(pred_cl).flatten()
    RESULTS.append(eval_metrics(act_u, pred_cl, 'CNN-LSTM'))

    # Learning curves comparison
    fig, ax = plt.subplots(figsize=(10, 4))
    for h, lbl in [(history_u, 'LSTM-Uni'), (history_gru, 'GRU'), (history_cl, 'CNN-LSTM')]:
        ax.plot(h.history['val_loss'], label=f'{lbl} val loss')
    ax.set_title('Validation Loss Comparison — DL Models')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
    savefig('22_dl_val_loss_comparison')

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(act_u,   color='#e63946', lw=1.5, label='Actual')
    ax.plot(pred_cl, color='#2a9d8f', ls='--', lw=1.5, label='CNN-LSTM')
    ax.legend(); ax.set_title('CNN-LSTM — Test Forecast')
    savefig('23_cnnlstm_forecast')
else:
    print('CNN-LSTM skipped.')

## Section 8 — Hyperparameter Optimisation

In [28]:
# ── 8.1  TSA Grid Search (SARIMA orders) ──────────────────────────────────────
if HAS_STATSMODELS:
    print('Grid search over SARIMA(p,1,q)(P,1,Q,12) — p,q ∈ {0,1,2}  P,Q ∈ {0,1}')
    best_aic_s, best_cfg_s = np.inf, None
    for p in range(3):
        for q in range(3):
            for P in range(2):
                for Q in range(2):
                    try:
                        m = SARIMAX(y_train, order=(p,1,q),
                                    seasonal_order=(P,1,Q,12),
                                    enforce_stationarity=False,
                                    enforce_invertibility=False).fit(disp=False)
                        if m.aic < best_aic_s:
                            best_aic_s = m.aic
                            best_cfg_s = (p,1,q,P,1,Q)
                    except:
                        pass
    print(f'Best SARIMA: order=({best_cfg_s[0]},{best_cfg_s[1]},{best_cfg_s[2]}) '
          f'seasonal=({best_cfg_s[3]},{best_cfg_s[4]},{best_cfg_s[5]},12)  AIC={best_aic_s:.2f}')
else:
    print('TSA grid search skipped.')

Grid search over SARIMA(p,1,q)(P,1,Q,12) — p,q ∈ {0,1,2}  P,Q ∈ {0,1}
Best SARIMA: order=(1,1,2) seasonal=(0,1,1,12)  AIC=822.60


In [29]:
# ── 8.2  Optuna Optimisation — LSTM ───────────────────────────────────────────
if HAS_OPTUNA and HAS_TF:
    def objective(trial):
        units   = trial.suggest_categorical('units',   [32, 64, 128])
        dropout = trial.suggest_float('dropout', 0.1, 0.4)
        lr      = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
        batch   = trial.suggest_categorical('batch', [16, 32, 64])

        model = build_lstm_uni(units=units, dropout=dropout, lr=lr)
        history = model.fit(
            X_tr_u, y_tr_u,
            validation_split=0.15,
            epochs=30, batch_size=batch,
            callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
            verbose=0
        )
        val_rmse = np.sqrt(min(history.history['val_loss']))
        return val_rmse

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=20, show_progress_bar=False)

    print('Best trial:', study.best_trial.params)
    print(f'Best val RMSE: {study.best_value:.6f}')

    # Retrain best model
    bp = study.best_trial.params
    lstm_best = build_lstm_uni(units=bp['units'], dropout=bp['dropout'], lr=bp['lr'])
    lstm_best.fit(
        X_tr_u, y_tr_u,
        validation_split=0.15,
        epochs=100, batch_size=bp['batch'],
        callbacks=CALLBACKS, verbose=0
    )
    pred_opt = lstm_best.predict(X_te_u, verbose=0)
    pred_opt = scaler_uni.inverse_transform(pred_opt).flatten()
    RESULTS.append(eval_metrics(act_u, pred_opt, 'LSTM-Optuna'))

    # Optuna visualisation
    try:
        import optuna.visualization.matplotlib as ovm
        fig = ovm.plot_optimization_history(study)
        savefig('24_optuna_history')
    except Exception:
        pass
elif HAS_TF:
    print('Optuna not installed — using default LSTM as best model.')
    lstm_best = lstm_uni
    pred_opt  = pred_u
else:
    print('Optuna+TF skipped.')

Best trial: {'units': 64, 'dropout': 0.14966826838018377, 'lr': 0.007679032290797027, 'batch': 16}
Best val RMSE: 0.029916


## Section 9 — Model Comparison

In [30]:
# ── 9.1  Leaderboard Table ────────────────────────────────────────────────────
if RESULTS:
    results_df = pd.DataFrame(RESULTS).sort_values('RMSE').reset_index(drop=True)
    print('\n=== MODEL LEADERBOARD ===')
    print(results_df.to_string(index=False, float_format='{:.4f}'.format))

    # ── 9.2  Bar Chart ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
        colors = sns.color_palette('muted', len(results_df))
        bars = ax.barh(results_df['Model'], results_df[metric], color=colors)
        ax.set_xlabel(metric)
        ax.set_title(f'{metric} — All Models')
        ax.invert_yaxis()
        for bar, val in zip(bars, results_df[metric]):
            ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center', fontsize=8)
    savefig('25_model_comparison_bars')

    # ── 9.3  Error Distribution Plots ────────────────────────────────────────
    # Collect predictions available in scope
    preds_available = {}
    if 'arima_fc' in dir(): preds_available['ARIMA']       = arima_fc.values[:len(y_test)]
    if 'sarima_fc' in dir(): preds_available['SARIMA']     = sarima_fc.values[:len(y_test)]
    if 'hw_fc' in dir(): preds_available['Holt-Winters']   = hw_fc.values[:len(y_test)]

    n_plots = len(preds_available)
    if n_plots > 0:
        fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
        if n_plots == 1: axes = [axes]
        for ax, (name, pred) in zip(axes, preds_available.items()):
            errors = y_test.values[:len(pred)] - pred
            ax.hist(errors, bins=25, color='#457b9d', edgecolor='white')
            ax.set_title(f'Error Distribution — {name}')
            ax.set_xlabel('Residual (°C)')
        savefig('26_error_distributions')
else:
    print('No results yet — run model cells first.')


=== MODEL LEADERBOARD ===
                  Model   RMSE    MAE      R2  Accuracy(%)
           Holt-Winters 0.3954 0.3209  0.9915      90.7614
                    VAR 0.4456 0.3628  0.9892      89.6057
         ARIMA(2, 1, 3) 0.4897 0.4015  0.9869      88.5571
SARIMA(1,1,1)(1,1,1,12) 0.4938 0.4117  0.9867      88.4614
            SARIMAX+EWS 0.5047 0.4237  0.9861      88.2070
            LSTM-Optuna 0.5350 0.4440  0.9844      87.4979
                    GRU 0.8631 0.7513  0.9593      79.8323
        LSTM-Univariate 0.9101 0.7382  0.9548      78.7332
                   VECM 2.0584 1.6602  0.7687      51.9015
               CNN-LSTM 4.7720 4.0283 -0.2433       0.0000
      LSTM-Multivariate 5.3000 4.3641 -0.5337       0.0000


## Section 10 — Hybrid Ensemble Modeling

In [31]:
# ── 10.1  Collect Available Test Predictions ──────────────────────────────────
all_test_preds = {}

if 'arima_fc'  in dir() and len(arima_fc)  == n_test: all_test_preds['ARIMA']  = arima_fc.values
if 'sarima_fc' in dir() and len(sarima_fc) == n_test: all_test_preds['SARIMA'] = sarima_fc.values
if 'hw_fc'     in dir() and len(hw_fc)     == n_test: all_test_preds['HW']     = hw_fc.values

if HAS_TF:
    if 'pred_u'  in dir(): all_test_preds['LSTM-Uni'] = pred_u[:n_test]
    if 'pred_gru' in dir(): all_test_preds['GRU']     = pred_gru[:n_test]
    if 'pred_cl'  in dir(): all_test_preds['CNN-LSTM'] = pred_cl[:n_test]

if len(all_test_preds) < 2:
    print('Not enough models for ensemble — run more model cells.')
else:
    # Align lengths
    min_len = min(len(v) for v in all_test_preds.values())
    min_len = min(min_len, len(y_test))
    y_ens   = y_test.values[:min_len]
    stack   = np.column_stack([v[:min_len] for v in all_test_preds.values()])
    names   = list(all_test_preds.keys())

    # ── 10.2  Inverse-RMSE Weighted Ensemble ─────────────────────────────────
    rmse_vals = np.array([rmse(y_ens, stack[:, i]) for i in range(stack.shape[1])])
    weights   = (1 / (rmse_vals + 1e-9))
    weights  /= weights.sum()

    print('Ensemble weights (inv-RMSE):')
    for name, w in zip(names, weights):
        print(f'  {name:<20} {w:.4f}')

    weighted_ens = stack @ weights
    RESULTS.append(eval_metrics(y_ens, weighted_ens, 'Weighted Ensemble'))

    # ── 10.3  Stacked Ensemble (Ridge meta-learner) ───────────────────────────
    from sklearn.linear_model import Ridge
    half = min_len // 2
    meta_X_tr, meta_y_tr = stack[:half],  y_ens[:half]
    meta_X_te, meta_y_te = stack[half:],  y_ens[half:]

    ridge = Ridge(alpha=1.0)
    ridge.fit(meta_X_tr, meta_y_tr)
    stacked_pred = ridge.predict(meta_X_te)
    RESULTS.append(eval_metrics(meta_y_te, stacked_pred, 'Stacked Ensemble (Ridge)'))

    print('\nRidge meta-learner coefficients:')
    for name, coef in zip(names, ridge.coef_):
        print(f'  {name:<20} {coef:.4f}')

    # ── 10.4  Uncertainty-Aware Ensemble ─────────────────────────────────────
    ens_mean = stack.mean(axis=1)
    ens_std  = stack.std(axis=1)
    lower    = ens_mean - 1.96 * ens_std
    upper    = ens_mean + 1.96 * ens_std

    idx_plot = np.arange(min_len)
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(idx_plot, y_ens,        color='#e63946', lw=1.5, label='Actual', zorder=5)
    ax.plot(idx_plot, weighted_ens, color='#2a9d8f', ls='--', lw=1.5, label='Weighted Ensemble')
    ax.fill_between(idx_plot, lower, upper, alpha=0.25, color='#2a9d8f', label='±1.96σ Uncertainty')
    ax.legend(); ax.set_title('Uncertainty-Aware Weighted Ensemble Forecast')
    savefig('27_ensemble_forecast')

    # ── 10.5  Updated Leaderboard ─────────────────────────────────────────────
    results_df = pd.DataFrame(RESULTS).sort_values('RMSE').reset_index(drop=True)
    print('\n=== FINAL LEADERBOARD ===')
    print(results_df.to_string(index=False, float_format='{:.4f}'.format))

Ensemble weights (inv-RMSE):
  ARIMA                0.2253
  SARIMA               0.2234
  HW                   0.2791
  LSTM-Uni             0.1212
  GRU                  0.1278
  CNN-LSTM             0.0231

Ridge meta-learner coefficients:
  ARIMA                0.3702
  SARIMA               0.4227
  HW                   0.1345
  LSTM-Uni             -0.2158
  GRU                  0.2917
  CNN-LSTM             0.1695

=== FINAL LEADERBOARD ===
                   Model   RMSE    MAE      R2  Accuracy(%)
Stacked Ensemble (Ridge) 0.3066 0.2545  0.9947      92.7275
            Holt-Winters 0.3954 0.3209  0.9915      90.7614
                     VAR 0.4456 0.3628  0.9892      89.6057
          ARIMA(2, 1, 3) 0.4897 0.4015  0.9869      88.5571
 SARIMA(1,1,1)(1,1,1,12) 0.4938 0.4117  0.9867      88.4614
             SARIMAX+EWS 0.5047 0.4237  0.9861      88.2070
       Weighted Ensemble 0.5335 0.4614  0.9845      87.5347
             LSTM-Optuna 0.5350 0.4440  0.9844      87.4979
         

## Section 11 — Future Forecasting & Risk Interpretation

In [32]:
# ── 11.1  12-Month and 24-Month Ahead Forecasts ───────────────────────────────
HORIZON = 24
future_dates = pd.date_range(df.index[-1] + pd.DateOffset(months=1),
                              periods=HORIZON, freq='MS')

future_preds = {}

# ── SARIMA ────────────────────────────────────────────────────────────────────
if HAS_STATSMODELS and 'sarima_model' in dir():
    sarima_future = SARIMAX(
        df['temperature'], order=(1,1,1), seasonal_order=(1,1,1,12),
        enforce_stationarity=False, enforce_invertibility=False
    ).fit(disp=False).forecast(steps=HORIZON)
    future_preds['SARIMA'] = sarima_future.values

# ── Holt-Winters ──────────────────────────────────────────────────────────────
if HAS_STATSMODELS:
    hw_full = ExponentialSmoothing(
        df['temperature'], trend='add', seasonal='add', seasonal_periods=12
    ).fit(optimized=True)
    future_preds['Holt-Winters'] = hw_full.forecast(HORIZON).values

# ── LSTM (rolling one-step ahead) ────────────────────────────────────────────
if HAS_TF and 'lstm_best' in dir():
    seed_seq = temp_scaled[-SEQ_LEN:].reshape(1, SEQ_LEN, 1)
    lstm_future = []
    for _ in range(HORIZON):
        pred_step = lstm_best.predict(seed_seq, verbose=0)[0, 0]
        lstm_future.append(pred_step)
        seed_seq = np.roll(seed_seq, -1, axis=1)
        seed_seq[0, -1, 0] = pred_step
    lstm_future_inv = scaler_uni.inverse_transform(
        np.array(lstm_future).reshape(-1, 1)
    ).flatten()
    future_preds['LSTM'] = lstm_future_inv

# ── Ensemble future ───────────────────────────────────────────────────────────
if len(future_preds) >= 2:
    ens_future = np.column_stack(list(future_preds.values())).mean(axis=1)
    future_preds['Ensemble'] = ens_future

# ── 11.2  Plot Futures ────────────────────────────────────────────────────────
colors_future = {'SARIMA': '#457b9d', 'Holt-Winters': '#2a9d8f',
                 'LSTM': '#f4a261', 'Ensemble': '#e63946'}

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index[-60:], df['temperature'].iloc[-60:],
        color='black', lw=1.5, label='Historical')
for name, vals in future_preds.items():
    ax.plot(future_dates[:len(vals)], vals,
            color=colors_future.get(name, 'gray'),
            ls='--', lw=1.5, label=f'{name} Forecast')
ax.axvline(df.index[-1], color='grey', ls=':', lw=1)
ax.legend(fontsize=9)
ax.set_title(f'{HORIZON}-Month Ahead Forecast — All Models')
ax.set_ylabel('Temperature (°C)')
savefig('28_future_forecasts')

# ── 11.3  Instability Risk Heatmap ────────────────────────────────────────────
if 'ens_future' in dir():
    ens_series = pd.Series(ens_future, index=future_dates)
    trend_delta = ens_series.diff().fillna(0)
    risk_idx    = (trend_delta - trend_delta.mean()) / (trend_delta.std() + 1e-9)

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(future_dates, ens_future, color='#e63946', lw=1.5, label='Ensemble Forecast')
    axes[0].legend()
    axes[1].bar(future_dates, risk_idx,
                color=['#e63946' if r > 1 else '#2a9d8f' for r in risk_idx],
                label='Normalised Instability Risk')
    axes[1].axhline(1, color='red', ls='--', lw=0.8, label='Risk Threshold')
    axes[1].legend()
    axes[1].set_ylabel('Risk Score (σ)')
    fig.suptitle('Future Forecast + Climate Instability Risk Identification', fontsize=13)
    savefig('29_instability_risk')

    high_risk = future_dates[risk_idx > 1]
    print(f'\nHigh-risk months (risk > 1σ): {list(high_risk.strftime("%Y-%m"))}')


High-risk months (risk > 1σ): ['2026-04', '2026-05', '2026-06', '2027-04', '2027-05', '2027-06']


## Section 12 — Conclusions

In [35]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('=' * 65)
print('  CLIMATE TIPPING POINT PREDICTION — FINAL SUMMARY')
print('=' * 65)

if RESULTS:
    results_df = pd.DataFrame(RESULTS).sort_values('RMSE').reset_index(drop=True)
    best_model = results_df.iloc[0]
    print(f'\nBest Model : {best_model["Model"]}')
    print(f'  RMSE     : {best_model["RMSE"]:.4f}')
    print(f'  MAE      : {best_model["MAE"]:.4f}')
    print(f'  R²       : {best_model["R2"]:.4f}')
    print(f'  Accuracy : {best_model["Accuracy(%)"]:.2f}%')

    # Final comparison plot
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot = results_df.set_index('Model')[['RMSE', 'MAE']]
    pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='black', width=0.7)
    ax.set_title('Final Model Benchmark — RMSE & MAE', fontsize=13)
    ax.set_ylabel('Error (°C)')
    ax.tick_params(axis='x', rotation=35)
    savefig('30_final_benchmark')

print('\nAll figures saved to ./figures/  |  Notebook complete ✓')

  CLIMATE TIPPING POINT PREDICTION — FINAL SUMMARY

Best Model : Stacked Ensemble (Ridge)
  RMSE     : 0.3066
  MAE      : 0.2545
  R²       : 0.9947
  Accuracy : 92.73%

All figures saved to ./figures/  |  Notebook complete ✓


---
**End of Notebook**

*To use real ERA5 data: replace the synthetic data block in Section 2 with `xarray`/`cdsapi` loading code. All downstream sections are data-agnostic.*